# CoChem-CORE: Stage 0.0 Initialization

This master notebook integrates the `CoChem-UNITY` GUI (if provisioned) and securely invokes the `setup.py` global orchestrator. It parses the hardware limits and maps the micro-silo pathways required for downstream prediction and geometry escalations.

**Instructions:** Execute the cell below. Follow the GUI prompts to build the deployment manifest, then confirm to trigger the 7-phase installation sequence.

In [ ]:
import os
import sys
import subprocess
from IPython.display import display, clear_output
import ipywidgets as widgets

# Determine if the Unity Frontend exists in the local workspace
UNITY_AVAILABLE = os.path.exists('cochem_unity_installer.py')

def launch_orchestrator(b=None):
    """Invokes the master setup.py and streams the output directly to the Jupyter Cell."""
    clear_output()
    print("\033[96m\033[1m▶ Launching CoChem Master Orchestrator...\033[0m\n")
    
    if not os.path.exists('setup.py'):
        print("\033[91m❌ FATAL: setup.py not found in the root directory.\033[0m")
        return
        
    # Safely stream the output of the 7-phase orchestration process
    process = subprocess.Popen([sys.executable, 'setup.py'], 
                               stdout=subprocess.PIPE, 
                               stderr=subprocess.STDOUT, 
                               text=True, 
                               bufsize=1)
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
    process.stdout.close()
    process.wait()

if UNITY_AVAILABLE:
    try:
        from cochem_unity_installer import UnityInstaller
        installer = UnityInstaller()
        
        # Overwrite the default confirm button to directly chain into the Setup Sequence
        def unified_generate_and_launch(b):
            installer.generate_manifest(b)
            launch_orchestrator()
            
        installer.btn_confirm.on_click(unified_generate_and_launch)
        installer.render()
    except ImportError:
        print("Unity Installer dependencies missing. Reverting to headless orchestrator.")
        launch_orchestrator()
else:
    print("CoChem-UNITY dashboard not detected. Initializing headless CoChem-CORE build.")
    launch_orchestrator()